# Using the IBM dataset with Kosh and Pytorch


Assumes you are on LC machine.

In [1]:
import os
import kosh

# Make sure local file is new sql file
kosh_example_sql_file = "kosh_example.sql"

In [2]:
from  kosh import KoshStore
import os

# connect to store
store = KoshStore(engine="sina", username=os.environ["USER"], sql='sql', db_path=kosh_example_sql_file)

['4a46f9facba942058be4d007546292e3']


In [3]:
# Add MNST datasets to store
from sina.utils import DataRange
train_set = store.search(project="IBM", taperThresh=DataRange(.6), skewThresh=DataRange(.6), ids_only=True)
print(len(train_set))
test_set = list(set(store.search(project="IBM", ids_only=True)).symmetric_difference(train_set))

143


In [4]:
ds = store.open(train_set[9])
#print(ds)
h5 = ds.search(mime_type="hdf5")[0].open()
print(type(h5))
h5["node/metrics_0"].shape

<class 'kosh.loaders.core.KoshHDF5Object'>


(31546, 10323)

In [ ]:
import numpy
from torch.utils.data import Dataset
import aml_dmt

class IBMDataset(Dataset):
    def __init__(self, kosh_datasets, label_metric, features_metrics, symmetry=2, history_length=100):
        Dataset.__init__(self)
        # open and store datasets from kosh
        self.datasets = [store.open(ds) for ds in kosh_datasets]
        self.label_metric = label_metric
        self.datasets_length = []
        self.symmetry = symmetry
        self.features_metrics = features_metrics
        self.history_length = history_length
        for ds in self.datasets:
            self.datasets_length += [self.dataset_length(ds),]

    def dataset_length(self, ds):
        n = 0
        bad = ds.bad_nodes[self.label_metric]
        for node in bad:
            n += len(aml_dmt.ibm.string2indices(bad[node]))
        return n * self.symmetry
        
    def __len__(self):
        n = 0
        for length in self.datasets_length:
            n += length
        return n
        
    def __getitem__(self, key):
        # figure out which dataset to use
        total = 0
        for ds_index, length in enumerate(self.datasets_length):
            total += length
            if total > key:
                break
        # offset 0 means first node in this dataset
        offset = total - length
        # index in dataset
        node_index = (key - offset) // self.symmetry
        # bad node type or good node type
        node_type = key % self.symmetry
        # sort the nodes so it is always the same node for each index
        node_keys = sorted([int(k) for k in self.datasets[ds_index].bad_nodes[self.label_metric]])
        index_in_bads = 0
        bads = self.datasets[ds_index].bad_nodes[self.label_metric]
        for node in bads:
            indices = aml_dmt.ibm.string2indices(bads[node])
            n = len(indices)
            if index_in_bads + n < node_index:
                index_in_bads += n
            else:
                time_index = indices[index_in_bads + n - node_index]
        node = int(node)
        #time_index = index_in_bads
        h5 = self.datasets[ds_index].search(mime_type="hdf5")[0].open()
        ntimes = h5["node/metrics_0"].shape[0]
        if node_type == 0:  # we want a bad node
            # nothing to do we're good
            pass
        else:
            # skip a few nodes
            node += node_type
            if node > ntimes: # make sure it is the valid range
                node -= ntimes
            while str(node) in bads:
                # skip a few nodes
                node += self.symmetry
                if node > ntimes: # make sure it is the valid range
                    node -= ntimes
        out = None
        for metric in self.features_metrics:
            print(metric, time_index, node)
            data = h5["node/{}".format(metric)][time_index-self.history_length+1:time_index+1, node]
            data = data.reshape((data.shape[0],1))
            # We probably should normalize here...
            if out is None:
                out = data
            else:
                out = numpy.hstack((out, data))
        # Numpy to torch?
        return out,numpy.array(node_type)

In [28]:
ibm = IBMDataset(train_set, "metrics_3", ["metrics_20", "metrics_23"], history_length=20)

ibm[3][0].shape

metrics_20 47467 3692
Read: (47915, 10323) (20,)
metrics_23 47467 3692
Read: (47915, 10323) (20,)


(20, 2)

In [29]:
from torch.utils.data import DataLoader

dataloader = DataLoader(ibm, batch_size=4,
                        shuffle=True, num_workers=4)



In [ ]:
# Stolen from pytorch example mnist
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


use_cuda = False
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 20, 5, 1)
        self.conv2 = nn.Conv2d(20, 50, 5, 1)
        self.fc1 = nn.Linear(4*4*50, 500)
        self.fc2 = nn.Linear(500, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2, 2)
        x = x.view(-1, 4*4*50)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

    
def train(args, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % args.log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))

def test(args, model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item() # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True) # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

    # Training settings
import argparse
parser = argparse.ArgumentParser(description='PyTorch MNIST Example')
parser.add_argument('--batch-size', type=int, default=64, metavar='N',
                    help='input batch size for training (default: 64)')
parser.add_argument('--test-batch-size', type=int, default=1000, metavar='N',
                    help='input batch size for testing (default: 1000)')
parser.add_argument('--epochs', type=int, default=10, metavar='N',
                    help='number of epochs to train (default: 10)')
parser.add_argument('--lr', type=float, default=0.01, metavar='LR',
                    help='learning rate (default: 0.01)')
parser.add_argument('--momentum', type=float, default=0.5, metavar='M',
                    help='SGD momentum (default: 0.5)')
parser.add_argument('--no-cuda', action='store_true', default=False,
                    help='disables CUDA training')
parser.add_argument('--seed', type=int, default=1, metavar='S',
                    help='random seed (default: 1)')
parser.add_argument('--log-interval', type=int, default=10, metavar='N',
                    help='how many batches to wait before logging training status')

parser.add_argument('--save-model', action='store_true', default=False,
                    help='For Saving the current Model')
args = parser.parse_args(())
use_cuda = not args.no_cuda and torch.cuda.is_available()
torch.manual_seed(args.seed)

device = torch.device("cuda" if use_cuda else "cpu")
kwargs = {'num_workers': 1, 'pin_memory': True} if use_cuda else {}

train_loader = torch.utils.data.DataLoader(
    train_set.open(),
    batch_size=args.batch_size, shuffle=True, **kwargs)
test_loader = torch.utils.data.DataLoader(
    test_set.open(),
    batch_size=args.test_batch_size, shuffle=True, **kwargs)

model = Net().to(device)
optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=args.momentum)

for epoch in range(1, args.epochs + 1):
    train(args, model, device, train_loader, optimizer, epoch)
    test(args, model, device, test_loader)

